# 11C — Broad, sector-enriched, and combined seed propagation

This notebook compares three critical-minerals seed designs: the previously frozen six-firm broad-market seed specification, the untouched automated outputs from the sector-enriched supplement, and their union. No new human decision is introduced in this comparison.

The experiment separates two questions. `relatedness` asks whether a firm is connected to the theme at all. `signed_sensitivity` asks whether the disclosed mechanism implies a positive or negative response. An unclear direction therefore remains an exposure seed for relatedness but receives zero directional weight.

This stage uses no return or other outcome data. It freezes comparable embedding-based candidate scores before evaluation.

## 1. Environment and immutable paths

Load only the frozen LLM outputs and pre-existing 2024 asset embeddings.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
if HERE.name == 'quarterly_sec_pilot':
    PILOT_ROOT, GEO_ROOT, ROOT = HERE, HERE.parent, HERE.parents[1]
elif HERE.name == 'geopolitical_exposure':
    GEO_ROOT, ROOT = HERE, HERE.parent
    PILOT_ROOT = GEO_ROOT / 'quarterly_sec_pilot'
elif HERE.name == 'Asset_Embeddings':
    ROOT = HERE; GEO_ROOT = ROOT / 'geopolitical_exposure'; PILOT_ROOT = GEO_ROOT / 'quarterly_sec_pilot'
else:
    ROOT = (HERE / 'Projects' / 'Asset_Embeddings').resolve()
    GEO_ROOT = ROOT / 'geopolitical_exposure'; PILOT_ROOT = GEO_ROOT / 'quarterly_sec_pilot'

BASE_EXPOSURES = PILOT_ROOT / 'results' / 'extraction' / 'adjudicated_document_exposures.parquet'
ENRICHED_EXPOSURES = PILOT_ROOT / 'sector_supplement' / 'results' / 'document_exposures.parquet'
OUT = PILOT_ROOT / 'three_seed_specifications'
MANIFESTS = OUT / 'manifests'; RESULTS = OUT / 'results'; PROCESSED = OUT / 'processed'
for path in [MANIFESTS, RESULTS, PROCESSED]: path.mkdir(parents=True, exist_ok=True)
VINTAGES = ['2024q1','2024q2','2024q3','2024q4']
CUTOFFS = dict(zip(VINTAGES, pd.to_datetime(['2024-03-31','2024-06-30','2024-09-30','2024-12-31'])))
OUTCOMES_ACCESSED = False
print('Project root:', ROOT)
print('Outcomes accessed:', OUTCOMES_ACCESSED)

## 2. Freeze the existing extraction records

The broad arm reproduces the six-firm specification already frozen by the preceding notebook. The enriched arm uses every schema-valid LLM record exactly as generated, without manual filtering.

In [ ]:
def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert BASE_EXPOSURES.exists() and ENRICHED_EXPOSURES.exists()
broad_records = pd.read_parquet(BASE_EXPOSURES).query("review_decision == 'accept'").copy()
enriched_records = pd.read_parquet(ENRICHED_EXPOSURES).copy()
required = ['document_id','firm_id','available_date','mechanism','direction','directness',
            'exposure_strength','confidence','quote_valid']
for name, frame in [('broad', broad_records), ('sector_enriched', enriched_records)]:
    missing = sorted(set(required) - set(frame.columns))
    assert not missing, f'{name} missing columns: {missing}'
    assert frame.quote_valid.fillna(False).all(), f'{name} contains an invalid evidence quote'
    frame['firm_id'] = pd.to_numeric(frame.firm_id, errors='raise').astype(int)
    frame['available_date'] = pd.to_datetime(frame.available_date)
    frame['source_sample'] = name

extraction_audit = pd.DataFrame([
    {'seed_specification':'broad', 'records':len(broad_records), 'firms':broad_records.firm_id.nunique(),
     'source_sha256':file_sha256(BASE_EXPOSURES)},
    {'seed_specification':'sector_enriched', 'records':len(enriched_records), 'firms':enriched_records.firm_id.nunique(),
     'source_sha256':file_sha256(ENRICHED_EXPOSURES)},
])
extraction_audit.to_parquet(MANIFESTS / 'automated_extraction_input_audit.parquet', index=False)
display(extraction_audit)
assert broad_records.firm_id.nunique() == 7  # one accepted candidate remains below the frozen 0.30 seed threshold
assert enriched_records.firm_id.nunique() == 16

## 3. Apply one mechanical quarterly aggregation rule

For each vintage, retain the most recently available firm–mechanism disclosure. If multiple records share that date, retain the record with the largest confidence-weighted strength. The combined specification is formed before aggregation so duplicate evidence cannot be double-counted.

In [ ]:
direction_sign = {'positive':1.0, 'negative':-1.0, 'mixed':0.0, 'unclear':0.0}

def prepare_records(frame):
    x = frame.copy()
    x['relatedness_score'] = pd.to_numeric(x.exposure_strength) * pd.to_numeric(x.confidence)
    x['direction_sign'] = x.direction.map(direction_sign).fillna(0.0)
    x['signed_score'] = x.relatedness_score * x.direction_sign
    return x

broad_records = prepare_records(broad_records)
enriched_records = prepare_records(enriched_records)
combined_records = (pd.concat([broad_records, enriched_records], ignore_index=True)
                    .drop_duplicates(['document_id','firm_id','mechanism','direction','evidence_quote']))
record_sets = {'broad':broad_records, 'sector_enriched':enriched_records, 'combined':combined_records}

def latest_mechanism_panel(frame, specification):
    pieces = []
    for vintage, cutoff in CUTOFFS.items():
        eligible = frame.loc[frame.available_date.le(cutoff)].copy()
        eligible = eligible.sort_values(
            ['firm_id','mechanism','available_date','relatedness_score','document_id'])
        latest = eligible.drop_duplicates(['firm_id','mechanism'], keep='last').copy()
        latest['vintage'] = vintage; latest['seed_specification'] = specification
        pieces.append(latest)
    return pd.concat(pieces, ignore_index=True)

mechanism_seeds = pd.concat([latest_mechanism_panel(frame, name) for name, frame in record_sets.items()], ignore_index=True)
mechanism_seeds = mechanism_seeds.loc[mechanism_seeds.relatedness_score.ge(0.30)].copy()
overall_seeds = (mechanism_seeds.sort_values(['seed_specification','vintage','firm_id','relatedness_score'])
                 .drop_duplicates(['seed_specification','vintage','firm_id'], keep='last')
                 .copy())
overall_seeds['channel'] = 'critical_minerals_overall'
mechanism_seeds['channel'] = mechanism_seeds.mechanism
all_seeds = pd.concat([overall_seeds, mechanism_seeds], ignore_index=True)
all_seeds.to_parquet(PROCESSED / 'automated_quarterly_seed_panel.parquet', index=False)

seed_inventory = (overall_seeds.groupby(['seed_specification','vintage'], as_index=False)
                  .agg(seed_firms=('firm_id','nunique'), source_records=('document_id','nunique'),
                       positive=('direction',lambda x:int((x=='positive').sum())),
                       negative=('direction',lambda x:int((x=='negative').sum())),
                       zero_direction=('direction_sign',lambda x:int((x==0).sum()))))
seed_inventory.to_parquet(RESULTS / 'seed_inventory.parquet', index=False)
display(seed_inventory)
display(pd.DataFrame([{'broad_enriched_firm_overlap':len(set(broad_records.firm_id) & set(enriched_records.firm_id)),
                       'combined_eligible_seed_firms':overall_seeds.loc[overall_seeds.seed_specification.eq('combined'),'firm_id'].nunique()}]))

## 4. Load the quarterly asset embeddings

Both paper-style 10-dimensional RS-Binary and RS-L-Min embeddings are used. Every comparison uses the same model, quarter, universe, and scoring rule.

In [ ]:
embedding_frames = {}; embedding_audit = []
for vintage in VINTAGES:
    for model in ['rs_binary','rs_l_min']:
        path = ROOT / 'active_passive_extension' / 'benchmarks' / 'all_s12_control' / vintage / model / 'asset_embeddings_10d.parquet'
        frame = pd.read_parquet(path).sort_values('permco').reset_index(drop=True)
        dimensions = [c for c in frame.columns if c.startswith('embedding_')]
        assert len(dimensions) == 10 and not frame.permco.duplicated().any()
        frame['permco'] = pd.to_numeric(frame.permco, errors='raise').astype(int)
        embedding_frames[(vintage,model)] = frame
        for specification in ['broad','sector_enriched','combined']:
            quarter_seeds = set(overall_seeds.loc[(overall_seeds.vintage.eq(vintage)) &
                                                    (overall_seeds.seed_specification.eq(specification)),'firm_id'])
            present = quarter_seeds & set(frame.permco); missing = quarter_seeds - set(frame.permco)
            embedding_audit.append({'vintage':vintage,'model':model,'seed_specification':specification,
                                    'firms':len(frame),'dimensions':len(dimensions),
                                    'seed_firms':len(quarter_seeds),'seed_firms_present':len(present),
                                    'missing_seed_firms':len(missing),
                                    'missing_permcos':','.join(map(str,sorted(missing)))})
embedding_audit = pd.DataFrame(embedding_audit)
embedding_audit.to_parquet(MANIFESTS / 'embedding_audit.parquet', index=False)
display(embedding_audit)
assert embedding_audit.seed_firms_present.gt(0).all()
print('Missing firms are retained in the audit but cannot seed a holdings embedding in that quarter.')

## 5. Propagate every seed specification

Rows are L2-normalized, making their dot product cosine similarity. A candidate score is the seed-score-weighted average cosine similarity. Seed firms are flagged and excluded from subsequent incremental-discovery rankings.

In [ ]:
def normalized_matrix(frame, dimensions):
    matrix = frame[dimensions].to_numpy(dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms>0)

def propagate(frame, matrix, seeds, score_column, specification, vintage, model, channel):
    seed_lookup = frame.reset_index().merge(
        seeds[['firm_id',score_column]], left_on='permco', right_on='firm_id', how='inner')
    if seed_lookup.empty:
        return pd.DataFrame()
    weights = seed_lookup[score_column].to_numpy(dtype=float)
    denominator = np.abs(weights).sum()
    if denominator == 0:
        scores = np.zeros(len(frame))
    else:
        similarities = matrix @ matrix[seed_lookup['index'].to_numpy()].T
        scores = similarities @ weights / denominator
    out = frame[['permco'] + (['firm_name'] if 'firm_name' in frame else [])].copy()
    out['vintage']=vintage; out['embedding_model']=model; out['seed_specification']=specification
    out['channel']=channel; out['target']=score_column; out['propagated_score']=scores
    out['is_own_spec_seed']=out.permco.isin(seeds.firm_id)
    return out

score_frames = []
for (vintage, model), frame in embedding_frames.items():
    dimensions = [c for c in frame if c.startswith('embedding_')]
    matrix = normalized_matrix(frame, dimensions)
    for specification in ['broad','sector_enriched','combined']:
        selected = all_seeds[(all_seeds.vintage==vintage) & (all_seeds.seed_specification==specification)]
        for channel, group in selected.groupby('channel', sort=True):
            for target in ['relatedness_score','signed_score']:
                result = propagate(frame,matrix,group,target,specification,vintage,model,channel)
                if len(result): score_frames.append(result)
propagated_scores = pd.concat(score_frames, ignore_index=True)
combined_seed_firms = set(overall_seeds.loc[overall_seeds.seed_specification.eq('combined'),'firm_id'])
propagated_scores['is_any_seed'] = propagated_scores.permco.isin(combined_seed_firms)
propagated_scores['common_nonseed_rank'] = np.nan
nonseed_mask = ~propagated_scores.is_any_seed
propagated_scores.loc[nonseed_mask,'common_nonseed_rank'] = (propagated_scores.loc[nonseed_mask].groupby(
    ['vintage','embedding_model','seed_specification','channel','target']).propagated_score
    .rank(method='average', pct=True))
propagated_scores.to_parquet(RESULTS / 'three_specification_propagated_scores.parquet', index=False)
display(propagated_scores.groupby(['seed_specification','vintage','embedding_model','target'],as_index=False)
        .agg(rows=('permco','size'),channels=('channel','nunique'),seed_row_flags=('is_own_spec_seed','sum'),
             score_min=('propagated_score','min'),score_max=('propagated_score','max')).head(24))

## 6. Compare rankings on one common non-seed universe

All firms appearing in either seed sample are removed here. This prevents the larger enriched specification from appearing stronger merely because it directly labels more firms.

In [ ]:
common_nonseeds = propagated_scores.loc[~propagated_scores.is_any_seed].copy()
correlations = []
for keys, group in common_nonseeds.groupby(['vintage','embedding_model','channel','target']):
    wide = group.pivot(index='permco',columns='seed_specification',values='propagated_score')
    for left, right in [('broad','sector_enriched'),('broad','combined'),('sector_enriched','combined')]:
        if left in wide and right in wide:
            correlations.append(dict(zip(['vintage','embedding_model','channel','target'],keys),
                                     left=left,right=right,firms=len(wide),
                                     spearman=wide[left].corr(wide[right],method='spearman'),
                                     pearson=wide[left].corr(wide[right])))
rank_correlations = pd.DataFrame(correlations)
rank_correlations.to_parquet(RESULTS / 'cross_specification_rank_correlations.parquet', index=False)
top_candidates = (common_nonseeds.sort_values(
    ['vintage','embedding_model','seed_specification','channel','target','propagated_score'],
    ascending=[True,True,True,True,True,False])
    .groupby(['vintage','embedding_model','seed_specification','channel','target'],as_index=False).head(25))
top_candidates.to_parquet(RESULTS / 'top_25_common_nonseed_candidates.parquet', index=False)
display(rank_correlations.query("channel=='critical_minerals_overall'").sort_values(
    ['vintage','embedding_model','target','left']))
display(top_candidates.query("channel=='critical_minerals_overall' and target=='relatedness_score'")[
    ['vintage','embedding_model','seed_specification','permco','firm_name','propagated_score']].head(30))

## 7. Leave-one-seed-out internal recovery

Each seed firm is predicted from the remaining seeds in its own specification. This is an outcome-free diagnostic of whether known text discoveries cluster in embedding space. Channels with fewer than three nonzero directional seeds are not treated as estimable.

In [ ]:
loo_rows = []
for (vintage,model), frame in embedding_frames.items():
    dimensions=[c for c in frame if c.startswith('embedding_')]; matrix=normalized_matrix(frame,dimensions)
    row_index=pd.Series(frame.index.to_numpy(),index=frame.permco).to_dict()
    for specification in ['broad','sector_enriched','combined']:
        selected=all_seeds[(all_seeds.vintage==vintage)&(all_seeds.seed_specification==specification)]
        for channel, group in selected.groupby('channel',sort=True):
            for target in ['relatedness_score','signed_score']:
                usable=group.loc[group[target].ne(0) & group.firm_id.isin(row_index)].drop_duplicates('firm_id')
                if len(usable)<3: continue
                for held in usable.itertuples():
                    train=usable.loc[usable.firm_id.ne(held.firm_id)]
                    weights=train[target].to_numpy(float); denominator=np.abs(weights).sum()
                    prediction=(matrix[row_index[int(held.firm_id)]] @ matrix[[row_index[int(v)] for v in train.firm_id]].T @ weights / denominator)
                    loo_rows.append({'vintage':vintage,'embedding_model':model,'seed_specification':specification,
                                     'channel':channel,'target':target,'held_out_firm':int(held.firm_id),
                                     'actual':float(getattr(held,target)),'predicted':float(prediction),'training_seeds':len(train)})
loo = pd.DataFrame(loo_rows)
loo.to_parquet(RESULTS / 'leave_one_seed_out_predictions.parquet', index=False)
summary=[]
for keys,group in loo.groupby(['vintage','embedding_model','seed_specification','channel','target']):
    summary.append(dict(zip(['vintage','embedding_model','seed_specification','channel','target'],keys),
                        held_out_firms=len(group),mae=np.mean(np.abs(group.actual-group.predicted)),
                        spearman=group.actual.corr(group.predicted,method='spearman'),
                        pearson=group.actual.corr(group.predicted)))
loo_summary=pd.DataFrame(summary)
loo_summary.to_parquet(RESULTS / 'leave_one_seed_out_summary.parquet',index=False)
display(loo_summary.query("channel=='critical_minerals_overall'").sort_values(
    ['vintage','target','spearman'],ascending=[True,True,False]))

## 8. Freeze the outcome-free handoff

The manifest records the exact automated inputs and comparison design. The next notebook may attach outcomes, but it must not modify these scores after observing them.

In [ ]:
artifact_paths = [
    PROCESSED/'automated_quarterly_seed_panel.parquet', RESULTS/'seed_inventory.parquet',
    RESULTS/'three_specification_propagated_scores.parquet',
    RESULTS/'cross_specification_rank_correlations.parquet',
    RESULTS/'top_25_common_nonseed_candidates.parquet',
    RESULTS/'leave_one_seed_out_predictions.parquet', RESULTS/'leave_one_seed_out_summary.parquet']
inventory=[]
for path in artifact_paths:
    inventory.append({'artifact':path.name,'rows':len(pd.read_parquet(path)),
                      'bytes':path.stat().st_size,'sha256':file_sha256(path)})
inventory=pd.DataFrame(inventory); inventory.to_parquet(MANIFESTS/'output_inventory.parquet',index=False)
design={'stage':'three_seed_specification_propagation','seed_specifications':['broad','sector_enriched','combined'],
        'broad_input_sha256':file_sha256(BASE_EXPOSURES),'enriched_input_sha256':file_sha256(ENRICHED_EXPOSURES),
        'broad_seed_provenance':'previously frozen accepted records plus the pre-existing 0.30 provisional threshold',
        'sector_enriched_human_adjudication_used':False,'new_human_adjudication_used':False,
        'outcomes_accessed':False,'vintages':VINTAGES,
        'embedding_models':['rs_binary','rs_l_min'],'dimensions':10,
        'aggregation':'latest available firm-mechanism evidence by quarter; maximum relatedness mechanism for overall firm score',
        'propagation':'cosine similarity weighted by automated seed score and normalized by absolute seed weight',
        'incremental_evaluation_universe':'excludes union of all broad and enriched seed firms'}
design['sha256']=hashlib.sha256(json.dumps(design,sort_keys=True).encode()).hexdigest()
(MANIFESTS/'three_seed_specification_design.json').write_text(json.dumps(design,indent=2,sort_keys=True)+'\n')
display(inventory)
display(pd.DataFrame([design]))
assert not OUTCOMES_ACCESSED
print('Outcome-free three-specification handoff is frozen.')